# MUSTER MAB14: 새 CPU VM 자동 준비 후 Drive 자료 재생

**새 CPU Colab에서도 Run all 할 수 있습니다.** tiktoken0.14.0·nltk3.10.0·rank-bm250.2.2가 없으면 CPU 패키지만 준비하고, 공식 BPE 두 파일과 영문 punkt_tab의 바이트 해시를 검증해 자동 준비합니다. 기존 캐시도 검증 후 재사용합니다. 폴더만 있다고 통과하지 않습니다. 기존 SDK 설치·모델·Torch는 바꾸지 않습니다.

기존 Drive의 MAB v2 원 아카이브·영수증·원 계약·패키지를 FUSE로 직접 읽습니다. 공용 Drive API나 rclone을 쓰지 않습니다. GPU·모델·reader/judge 호출은0이며 baf7 원 휠은 로컬에 추출합니다. 재생 코드 ZIP과 출력 namespace는 이전 판과 같아 기존 완료 결과를 보존·재개합니다.

입장 관문은136작업·126저장소·1671문항 전체입니다. 원 source·QID·selection·context·request 기록을 검증한 뒤 최대4개 CPU 프로세스로 그래프200/400/줄이기만을 재생합니다. 미정·누락 점수·실패가 있으면 PARTIAL이며 PASS로 취급하지 않습니다.

**문맥 변경은 품질 향상이 아닙니다.** 원점수를 고정한 구조 진단이며 답 정확도·native QA 평가는 없습니다. 실제 완료 아카이브는 개발 호스트에서 읽지 못했으므로, 이 노트북은 아직 실제1671문항 재생 전입니다. 합성 자료의 함수 연결 검사와 입장 코드만 검증했습니다.

새 비공개 Drive 출력은 `MUSTER_mab_graph_replay_cpu_20261116`입니다. 마지막 셀에서 작은 `SUMMARY.JSON`을 표시하고 내려받습니다. 원문·벡터·모델 가중치는 이 노트북에 없습니다.


In [ ]:
from pathlib import Path
Path('/content/MUSTER_MAB14_CPU_REPLAY_ASSETS_READY.py').write_text('"""MAB14 CPU entry. Only pinned CPU tokenizers may install/download; no SDK/model/GPU."""\nimport _thread\nimport base64\nimport hashlib\nimport importlib.util\nimport importlib.metadata\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport subprocess\nimport urllib.request\nimport zipfile\n\nCODE_PAYLOAD = "..."\nCODE_SHA = "d63ad0a17b43fe723533fcf35ec88ed701707a49cd5860eae2c4fa3fc725abda"\n\n\ndef load(path, name):\n    spec = importlib.util.spec_from_file_location(name, path)\n    value = importlib.util.module_from_spec(spec)\n    sys.modules[name] = value\n    spec.loader.exec_module(value)\n    return value\n\n\ndef code():\n    data = base64.b64decode(CODE_PAYLOAD, validate=True)\n    if hashlib.sha256(data).hexdigest() != CODE_SHA:\n        raise RuntimeError("embedded CPU code differs")\n    root = Path("/content/muster_mab_cpu_code") / CODE_SHA[:16]\n    with zipfile.ZipFile(io.BytesIO(data)) as z:\n        manifest = json.loads(z.read("CODE_MANIFEST.json"))\n        if set(z.namelist()) != set(manifest["files"]) | {"CODE_MANIFEST.json"}:\n            raise RuntimeError("CPU code inventory differs")\n        for name, digest in manifest["files"].items():\n            p = Path(name)\n            if p.is_absolute() or ".." in p.parts:\n                raise RuntimeError("unsafe CPU code path")\n            raw = z.read(name)\n            if hashlib.sha256(raw).hexdigest() != digest:\n                raise RuntimeError("CPU code member differs")\n            dst = root / p\n            dst.parent.mkdir(parents=True, exist_ok=True)\n            dst.write_bytes(raw)\n    return root\n\n\nBPE_FILES = {\n    "cl100k_base": ("https://openaipublic.blob.core.windows.net/encodings/cl100k_base.tiktoken",\n                    "223921b76ee99bde995b7ff738513eef100fb51d18c93597a113bcffe865b2a7"),\n    "o200k_base": ("https://openaipublic.blob.core.windows.net/encodings/o200k_base.tiktoken",\n                   "446a9538cb6c348e3516120d7c08b09f57c36495e2acfffe59a5bf8b0cfb1a2d"),\n}\nPUNKT_URL = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/tokenizers/punkt_tab.zip"\nPUNKT_ZIP_SHA = "e57f64187974277726a3417ca6f181ec5403676c717672eef6a748a7b20e0106"\nPUNKT_FILES = {\n    "abbrev_types.txt": "92a3e070f43d9b4c5534758ca40ad7343b04e7e29bfe0c2eb658a39445a4f779",\n    "collocations.tab": "8e2da1225e4dd2cc9dba261ee231ccb134859e21b46006e7f472c5ee269af0cf",\n    "ortho_context.tab": "4bbcca25ed3d3f06c02402abf8419b9f033b8adc06e7b482eca4e45f81a5dc4c",\n    "sent_starters.txt": "f3f8535483e1dba487241b764945168123bca3209a9645e59acd1225dc76edac",\n}\n\n\ndef cpu_packages(version=None, install=None):\n    version = version or importlib.metadata.version\n    install = install or subprocess.run\n    pins = {"tiktoken": "0.14.0", "nltk": "3.10.0", "rank-bm25": "0.2.2"}\n    def actual(name):\n        try:\n            return version(name)\n        except importlib.metadata.PackageNotFoundError:\n            return None\n    def matches(name):\n        return actual(name) == pins[name] or (name == "nltk" and actual(name) == "3.10")\n    missing = [name + "==" + pin for name, pin in pins.items() if not matches(name)]\n    if missing:\n        print("고정 CPU 토크나이저 패키지 준비:", missing, flush=True)\n        install([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *missing], check=True)\n    if not all(matches(name) for name in pins):\n        raise RuntimeError("CPU tokenizer package versions differ after preparation")\n    return {name: actual(name) for name in pins}\n\n\ndef text_assets(base=None, fetch=None, ensure_packages=True):\n    versions = cpu_packages() if ensure_packages else {"fixture": "dependency install bypassed"}\n    reuse_external = base is None\n    base = Path(base or "/content/muster_cpu_text_assets_v1")\n    cache, assets = base / "tiktoken_cache", base / "muster"\n    english = assets / "nltk_data/tokenizers/punkt_tab/english"\n    def fetch_public(url):\n        print("공개 토크나이저 자산 준비:", url, flush=True)\n        with urllib.request.urlopen(url, timeout=120) as response:\n            return response.read()\n    fetch = fetch or fetch_public\n    def valid(path, digest):\n        return path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == digest\n    def save_verified(path, data, digest):\n        if hashlib.sha256(data).hexdigest() != digest:\n            raise RuntimeError("official tokenizer byte SHA mismatch: " + path.name)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        temp = path.with_name(path.name + ".writing")\n        temp.write_bytes(data)\n        os.replace(temp, path)\n    reused, downloaded = [], []\n    for name, (url, digest) in BPE_FILES.items():\n        key = hashlib.sha1(url.encode()).hexdigest()\n        target = cache / key\n        if valid(target, digest):\n            reused.append(name)\n            continue\n        candidates = []\n        if reuse_external:\n            candidates = [Path(x) / key for x in (os.environ.get("TIKTOKEN_CACHE_DIR"),\n                "/content/muster_assembly_parallel_assets/tiktoken_cache") if x]\n        found = next((x for x in candidates if valid(x, digest)), None)\n        save_verified(target, found.read_bytes() if found else fetch(url), digest)\n        (reused if found else downloaded).append(name)\n    if not all(valid(english / name, digest) for name, digest in PUNKT_FILES.items()):\n        old = Path(os.environ.get("MUSTER_ASSETS", str(Path.home() / ".local/share/muster"))) / "nltk_data/tokenizers/punkt_tab/english"\n        if reuse_external and all(valid(old / name, digest) for name, digest in PUNKT_FILES.items()):\n            for name, digest in PUNKT_FILES.items():\n                save_verified(english / name, (old / name).read_bytes(), digest)\n            reused.append("punkt_tab_english")\n        else:\n            data = fetch(PUNKT_URL)\n            if hashlib.sha256(data).hexdigest() != PUNKT_ZIP_SHA:\n                raise RuntimeError("official NLTK punkt_tab ZIP SHA mismatch")\n            with zipfile.ZipFile(io.BytesIO(data)) as z:\n                for name, digest in PUNKT_FILES.items():\n                    save_verified(english / name, z.read("punkt_tab/english/" + name), digest)\n            downloaded.append("punkt_tab_english")\n    else:\n        reused.append("punkt_tab_english")\n    if not all(valid(cache / hashlib.sha1(url.encode()).hexdigest(), digest) for url, digest in BPE_FILES.values()) or not all(\n            valid(english / name, digest) for name, digest in PUNKT_FILES.items()):\n        raise RuntimeError("CPU tokenizer asset validation incomplete")\n    config = {"TIKTOKEN_CACHE_DIR": str(cache), "MUSTER_ASSETS": str(assets), "NLTK_DATA": str(assets / "nltk_data")}\n    (base / "TEXT_ASSETS_READBACK.json").write_text(json.dumps({"status": "VERIFIED", "packages": versions,\n        "BPE_sha256": {name: value[1] for name, value in BPE_FILES.items()}, "english_sha256": PUNKT_FILES,\n        "punkt_zip_expected_sha256": PUNKT_ZIP_SHA, "reused": reused, "downloaded": downloaded,\n        "private_corpus_files": 0, "SDK_installed_or_modified": False, "model_GPU_calls": 0,\n        "config": config}, sort_keys=True, indent=2) + "\\n")\n    print("공개 CPU 토크나이저 자산 바이트 검증 완료. SDK·모델 설치0.", flush=True)\n    return config\n\n\ndef run():\n    if sys.platform != "linux" or not Path("/content").is_dir():\n        raise RuntimeError("this entry uses the existing Colab VM and mounted private Drive")\n    if not any(len(x.split()) > 1 and x.split()[1] == "/content/drive" for x in Path("/proc/mounts").read_text().splitlines()):\n        raise RuntimeError("Drive FUSE must be mounted")\n    root = code()\n    campaign = load(root / "metadata/campaign.py", "_mab_cpu_mirror")\n    module = load(root / "replay.py", "_mab_cpu_replay")\n    local = Path("/content/muster_mab_cpu_work") / CODE_SHA[:16]\n    drive = Path("/content/drive/MyDrive/MUSTER_mab_graph_replay_cpu_20261116/results") / CODE_SHA[:16]\n    archive = Path("/content/drive/MyDrive/MUSTER_mab_rank_20261115/results/v2/MAB_RANK_v2_1shard.tar.gz")\n    cfg = {"output_dir": str(local), "package_dir": "/content/drive/MyDrive/MUSTER_mab_rank_20261115",\n           "archive": str(archive), "receipt": str(archive) + ".receipt.json", "max_workers": 4,\n           "run_contract": str(archive.parent / "shard_1_of_1/RUN_CONTRACT.json"), **text_assets()}\n    print("MAB14 원1671문항·136작업 입장 검사 후 CPU 재생. GPU/모델/reader/judge 호출0.", flush=True)\n    print("비공개 원문은 Drive FUSE로만 읽고 새 SDK 설치나 공용 API를 쓰지 않습니다.", flush=True)\n    with campaign.campaign_owner(drive), campaign.CheckpointMirror(local, drive, on_error=_thread.interrupt_main) as mirror:\n        result = module.run_phase(cfg)\n        mirror.sync()\n    print("SUMMARY.JSON:", local / "SUMMARY.JSON", flush=True)\n    print("상태:", result["status"], "전체 문항 재생:", result["all_1671_rows_replayed"], flush=True)\n    print("문맥 변경은 품질 향상이 아닙니다. 미정·누락 점수는 PASS로 표시하지 않습니다.", flush=True)\n    return result, str(local / "SUMMARY.JSON")\n\n\nif __name__ == "__main__":\n    result, path = run()\n    print(json.dumps(result, ensure_ascii=False))\n', encoding='utf-8')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib.util
spec = importlib.util.spec_from_file_location('_mab_cpu_entry', '/content/MUSTER_MAB14_CPU_REPLAY_ASSETS_READY.py')
entry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(entry)
result, summary_path = entry.run()


In [ ]:
import json
from pathlib import Path
print(json.dumps(json.loads(Path(summary_path).read_text()), ensure_ascii=False, indent=1))
from google.colab import files
files.download(summary_path)
